<a href="https://colab.research.google.com/github/EmilioSantiago/Floreria-Back/blob/main/EmilioGabrielGarciaSantiago.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimentos con Perceptrón Multicapa (MLP) en Keras

**Objetivo:** modificar exclusivamente la arquitectura de una red neuronal MLP para observar cómo el número de capas ocultas y neuronas afecta el rendimiento en la clasificación del dataset Iris.

> Nota: los resultados pueden variar ligeramente entre ejecuciones por la inicialización aleatoria del entrenamiento. En este cuaderno se fijan semillas para que los resultados sean más estables.

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

# Semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# 1. Cargar el dataset Iris
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# 2. Preprocesamiento de datos (No modificar)
encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Función para entrenar y evaluar cada arquitectura

La función conserva el mismo procesamiento, compilación y entrenamiento. Lo único que cambia es la lista de capas ocultas.

In [2]:
def crear_modelo(neuronas_ocultas):
    model = Sequential()

    # Primera capa: obligatoriamente incluye input_shape=(4,)
    model.add(Dense(neuronas_ocultas[0], input_shape=(4,), activation="relu"))

    # Capas ocultas adicionales
    for n in neuronas_ocultas[1:]:
        model.add(Dense(n, activation="relu"))

    # Capa de salida obligatoria
    model.add(Dense(3, activation="softmax"))
    return model


def entrenar_y_evaluar(nombre, neuronas_ocultas):
    print("\n" + "=" * 70)
    print(nombre)
    print("Arquitectura oculta:", neuronas_ocultas)

    np.random.seed(42)
    tf.random.set_seed(42)

    model = crear_modelo(neuronas_ocultas)
    model.summary()

    model.compile(
        optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=100,
        batch_size=8,
        validation_split=0.1,
        verbose=0,
    )

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print("\n--- RESULTADOS DEL MODELO ---")
    print(f"Pérdida en el conjunto de prueba (Loss): {loss:.4f}")
    print(f"Precisión en el conjunto de prueba (Accuracy): {accuracy * 100:.2f}%")

    return {
        "Configuración": nombre,
        "Capas ocultas y neuronas": ", ".join([f"Capa {i+1}: {n}" for i, n in enumerate(neuronas_ocultas)]),
        "Loss": round(loss, 4),
        "Accuracy %": round(accuracy * 100, 2),
    }

## Modelo Base

Arquitectura oculta: **[16, 8]**

In [3]:
resultado_0 = entrenar_y_evaluar("Modelo Base", [16, 8])


Modelo Base
Arquitectura oculta: [16, 8]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)


--- RESULTADOS DEL MODELO ---
Pérdida en el conjunto de prueba (Loss): 0.0522
Precisión en el conjunto de prueba (Accuracy): 96.67%


## Experimento A (Profunda)

Arquitectura oculta: **[32, 16, 8, 4]**

In [4]:
resultado_1 = entrenar_y_evaluar("Experimento A (Profunda)", [32, 16, 8, 4])


Experimento A (Profunda)
Arquitectura oculta: [32, 16, 8, 4]


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            15 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 875 (3.42 KB)

 Trainable params: 875 (3.42 KB)

 Non-trainable params: 0 (0.00 B)


--- RESULTADOS DEL MODELO ---
Pérdida en el conjunto de prueba (Loss): 0.0276
Precisión en el conjunto de prueba (Accuracy): 100.00%


## Experimento B (Ancha)

Arquitectura oculta: **[128]**

In [5]:
resultado_2 = entrenar_y_evaluar("Experimento B (Ancha)", [128])


Experimento B (Ancha)
Arquitectura oculta: [128]


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027 (4.01 KB)

 Trainable params: 1,027 (4.01 KB)

 Non-trainable params: 0 (0.00 B)


--- RESULTADOS DEL MODELO ---
Pérdida en el conjunto de prueba (Loss): 0.0405
Precisión en el conjunto de prueba (Accuracy): 100.00%


## Experimento C (Minimalista)

Arquitectura oculta: **[2]**

In [6]:
resultado_3 = entrenar_y_evaluar("Experimento C (Minimalista)", [2])


Experimento C (Minimalista)
Arquitectura oculta: [2]


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19 (76.00 B)

 Trainable params: 19 (76.00 B)

 Non-trainable params: 0 (0.00 B)


--- RESULTADOS DEL MODELO ---
Pérdida en el conjunto de prueba (Loss): 0.5801
Precisión en el conjunto de prueba (Accuracy): 63.33%


## Tabla comparativa final

In [7]:
import pandas as pd

resultados = [resultado_0, resultado_1, resultado_2, resultado_3]
df_resultados = pd.DataFrame(resultados)
df_resultados

,Configuración,Capas ocultas y neuronas,Loss,Accuracy %
0,Modelo Base,"Capa 1: 16, Capa 2: 8",0.0522,96.67
1,Experimento A (Profunda),"Capa 1: 32, Capa 2: 16, Capa 3: 8, Capa 4: 4",0.0276,100.00
2,Experimento B (Ancha),Capa 1: 128,0.0405,100.00
3,Experimento C (Minimalista),Capa 1: 2,0.5801,63.33


## Pregunta de cierre

De acuerdo con los resultados, la mejor estructura es la que logra alta precisión usando la menor complejidad posible. Si dos modelos tienen una precisión muy parecida, conviene elegir el modelo más simple porque entrena más rápido, usa menos parámetros y reduce el riesgo de sobreajuste.

En este problema, el dataset Iris es pequeño y relativamente sencillo; por eso una red demasiado grande no siempre representa una ventaja importante. La red ancha puede alcanzar muy buena precisión, pero tiene muchas más neuronas. La red profunda también puede funcionar bien, aunque aumenta la complejidad. La red minimalista permite observar el límite: si su precisión baja de 90%, significa que ya no tiene suficiente capacidad para separar correctamente las tres clases.